# Prediction

Use existing databases to make predictions and perform interpolation.

In [1]:
import pprint

import cutcutcodec

from mendevi.models.nn import DNN
from mendevi.models.gpr import GPR
from mendevi.models.lr import LR

In [2]:
dataset = "svtav1_vs_rav1e_vs_aom.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Biaised linear regression

Predicts the output as a linear combination of the inputs, plus a constant.
It is a very fast model $O(n)$.

In [3]:
# model training
lr_model = LR(
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=["encoder"],  # optional, this feature provides one model per encoder
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster encoder=libaom-av1...
█ █ this cluster contains 2428 observations
█ █ 8.59 s
█ Fit on cluster encoder=librav1e...
█ █ this cluster contains 2473 observations
█ █ 60.79 ms
█ Fit on cluster encoder=libsvtav1...
█ █ this cluster contains 2472 observations
█ █ 105.26 ms
█ 1 min 46 s


In [4]:
# validation from dataset
lr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")
# print(lr_model.accuracy)  # for each cluster name, associate for each output label, the predicted and validation data

Validate 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster encoder=libaom-av1...
█ █ this cluster contains 596 observations
█ █ 280.91 ms
█ Validate on cluster encoder=librav1e...
█ █ this cluster contains 615 observations
█ █ 32.61 ms
█ Validate on cluster encoder=libsvtav1...
█ █ this cluster contains 613 observations
█ █ 32.31 ms
█ 3 min 50 s
MAPE of log_rate: 3.54%
RMS  of log_rate: 0.2951
MAPE of psnr: 11.65%
RMS  of psnr: 6.709
MAPE of ssim: 4.17%
RMS  of ssim: 0.05084
MAPE of rate: 66.81%
RMS  of rate: 1.593e+07


In [5]:
# prediction from a video
pred = lr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'Linear Regression for a predictive model'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster encoder=libsvtav1...
█ █ this cluster contains 1 observations
█ █ 11.75 ms
█ 51.56 ms


{'log_rate': [6.372843265533447],
 'psnr': [34.86414337158203],
 'ssim': [0.9203274846076965]}


## Gaussian process regression

Very slow model $O(n^3)$.

In [6]:
# model training
gpr_model = GPR(  # no aggregation because it fits only on continuous parameters
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= .5",  # two slow on all data
)

Fit 'Gaussian Process Regression predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster effort=slow, encoder=libaom-av1...
█ █ this cluster contains 509 observations
█ █ KeyboardInterrupt l673 ()
█ KeyboardInterrupt l673 ()


KeyboardInterrupt: 

In [ ]:
# validation from dataset
gpr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

In [ ]:
# prediction from a video
pred = gpr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

## Deep neuronal network

Medium speed model, $O(n)$.

In [ ]:
# model training
dnn_model = DNN(  # see parameters niter, learning_rate and total
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=[],  # optional, this feature force a single common model
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

In [ ]:
# validation from dataset
dnn_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

In [ ]:
# prediction from a video
pred = dnn_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)